# CMSC 173 &middot; Module 13 &middot; Example 3
## Attention, by hand

Three tokens, cat, sat and mat, each with a query, a key and a value. Score, scale, softmax, weighted sum. Then one head of BERT, a real model, reading the word &ldquo;it&rdquo;.

**How to use this notebook.** Runtime &rarr; Run all. Every number printed below is on the
slides (NumPy; BERT through Hugging Face transformers; a newer version can differ slightly for the trained networks);
change something and see what moves.

**The three tokens.** In a real model the queries, keys and values come from learned weights; here they are written down directly.

In [ ]:
import numpy as np

tokens = ["cat", "sat", "mat"]
Q = np.array([[1, 0, 1, 0], [2, 0, 1, 1], [1, 0, 0, 1]])   # one query per token
K = np.array([[1, 0, 1, 0], [0, 1, 0, 1], [1, 0, 0, 1]])   # one key per token
V = np.array([[1, 0], [0, 1], [1, 1]])                     # one value per token

**Attention for &ldquo;sat&rdquo;.** Dot its query with every key, divide by $\sqrt{4}$, softmax, then add the values, each times its weight.

In [ ]:
q = Q[1]
scores = np.array([np.dot(q, k) for k in K]) / np.sqrt(4)      # 1.5 0.5 1.5
weights = np.exp(scores) / np.exp(scores).sum()                # 0.4223 0.1554 0.4223
z = sum(weights[j] * V[j] for j in range(3))                   # (0.8446, 0.5777)
print(scores, weights.round(4), z.round(4))

**All three tokens.** The same three steps for every query; each row of weights adds to 1.

In [ ]:
def attention(q):
    s = np.array([np.dot(q, k) for k in K]) / np.sqrt(len(q))
    a = np.exp(s) / np.exp(s).sum()
    return a, sum(a[j] * V[j] for j in range(len(V)))

for t, q in zip(tokens, Q):
    a, z = attention(q)
    print(t, a.round(4), z.round(4))
# cat [0.5065 0.1863 0.3072] [0.8137 0.4935]
# sat [0.4223 0.1554 0.4223] [0.8446 0.5777]
# mat [0.2741 0.2741 0.4519] [0.7259 0.7259]

**Why divide by $\sqrt{d_k}$.** 100,000 dot products of random lists of 64 numbers spread out by about $\sqrt{64} = 8$; dividing by 8 brings the spread back to about 1.

In [ ]:
rng = np.random.default_rng(173)
dots = (rng.standard_normal((100_000, 64)) * rng.standard_normal((100_000, 64))).sum(axis=1)
print(round(dots.std(), 2), round((dots / 8).std(), 2))        # 7.96 0.99

**One head of a real model.** BERT (Devlin et al., 2019) has 12 layers of 12 attention heads. Head 11 of layer 9 sends &ldquo;it&rdquo; to the noun it stands for. The download is about 440 MB.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()                                  # hide the download report

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True, attn_implementation="eager").eval()
for why in ("tired", "dirty"):
    enc = tok(f"the cat sat on the mat because it was {why}", return_tensors="pt")
    words = tok.convert_ids_to_tokens(enc["input_ids"][0])
    with torch.no_grad():
        att = bert(**enc).attentions[9 - 1][0, 11 - 1, words.index("it")]   # layer 9, head 11
    print(why, "cat", round(float(att[words.index("cat")]), 2), "mat", round(float(att[words.index("mat")]), 2))
# tired cat 0.88 mat 0.01
# dirty cat 0.58 mat 0.23

**Try it.** &ldquo;the dog chased the cat because it was scared&rdquo;: who was scared? Where does this head send &ldquo;it&rdquo;? One head is a clue, not understanding.